# Sector Expenditure Profiling
Owner: Razz  
Table: `pondolens.dev_bronze.sector_expenditure`  
Baseline: **414 rows**

**Goal:** check the data for Q1 — where government spending goes across sectors — and identify Silver treatments.

**Row meaning:** source publication × expenditure year × stage (`price_basis`) × accounting basis × sector × particulars. Amounts are in **million pesos**.

**Scope:** retain actual, program, proposed, raw values, and all source/load metadata. Use only actual data for Silver layer.

## 1. Structure and coverage
Check column types, sample rows, and years/stages available in each source.

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.sector_expenditure;

In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.sector_expenditure
LIMIT 20;

In [0]:
%sql
SELECT
    source_name,
    price_basis,
    accounting_basis,
    unit_amount,
    year,
    COUNT(*) AS row_count
FROM pondolens.dev_bronze.sector_expenditure
GROUP BY source_name, price_basis, accounting_basis, unit_amount, year
ORDER BY source_name, price_basis, year;


In [0]:
%sql
SELECT sector, COUNT(*) AS row_count,
    COUNT(DISTINCT NULLIF(TRIM(particulars), '')) AS nonblank_particulars,
    COUNT_IF(NULLIF(TRIM(particulars), '') IS NULL) AS blank_particulars
FROM pondolens.dev_bronze.sector_expenditure
GROUP BY sector
ORDER BY sector;


## 2. Data quality
Check required fields, numeric parsing, dash values, and duplicates. `TRY_CAST` returns null when a value cannot be parsed; it keeps the query running.

The duplicate query checks the key used to identify comparable records. **No rows means no repeated keys were found.**

In [0]:
%sql
WITH checked AS (
    SELECT *,
        TRY_CAST(year AS INT) AS year_value,
        TRY_CAST(source_page AS INT) AS page_value,
        TRY_CAST(REPLACE(expenditure_amount, ',', '') AS DECIMAL(20, 2)) AS amount_value,
        TRY_CAST(REPLACE(percent_distribution, '%', '') AS DECIMAL(10, 4)) AS percent_value
    FROM pondolens.dev_bronze.sector_expenditure
)
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(NULLIF(TRIM(sector), '') IS NULL) AS missing_sector,
    COUNT_IF(NULLIF(TRIM(particulars), '') IS NULL) AS missing_particulars,
    COUNT_IF(NULLIF(TRIM(price_basis), '') IS NULL) AS missing_price_basis,
    COUNT_IF(NULLIF(TRIM(accounting_basis), '') IS NULL) AS missing_accounting_basis,
    COUNT_IF(NULLIF(TRIM(unit_amount), '') IS NULL) AS missing_unit,
    COUNT_IF(year_value IS NULL) AS invalid_year,
    COUNT_IF(page_value IS NULL) AS invalid_source_page,
    COUNT_IF(amount_value IS NULL) AS invalid_amount,
    COUNT_IF(percent_value IS NULL) AS invalid_percent,
    COUNT_IF(amount_value = 0) AS zero_amount,
    COUNT_IF(amount_value < 0) AS negative_amount,
    COUNT_IF(TRIM(expenditure_amount) IN ('-', '--')) AS dash_amount,
    COUNT_IF(NULLIF(TRIM(_rescued_data), '') IS NOT NULL) AS rescued_rows
FROM checked;


In [0]:
%sql
SELECT year, price_basis, accounting_basis, unit_amount, sector,
    NULLIF(TRIM(particulars), '') AS particulars,
    COUNT(*) AS row_count
FROM pondolens.dev_bronze.sector_expenditure
GROUP BY year, price_basis, accounting_basis, unit_amount, sector,
    NULLIF(TRIM(particulars), '')
HAVING COUNT(*) > 1
ORDER BY row_count DESC;


## 3. Reconcile totals and check duplicate join keys
Check the GRAND TOTAL join key before comparing percentages. The first query returns only groups with missing or repeated GRAND TOTAL rows; an empty result is expected when each group has one total.

In [0]:
%sql
SELECT source_name, year, price_basis, accounting_basis, unit_amount,
    COUNT_IF(UPPER(TRIM(sector)) = 'GRAND TOTAL') AS grand_total_rows
FROM pondolens.dev_bronze.sector_expenditure
GROUP BY source_name, year, price_basis, accounting_basis, unit_amount
HAVING COUNT_IF(UPPER(TRIM(sector)) = 'GRAND TOTAL') <> 1;


In [0]:
%sql
WITH amounts AS (
    SELECT *,
        TRY_CAST(REPLACE(TRIM(expenditure_amount), ',', '') AS DECIMAL(20, 2)) AS amount,
        TRY_CAST(REPLACE(TRIM(percent_distribution), '%', '') AS DECIMAL(10, 4)) AS reported_pct
    FROM pondolens.dev_bronze.sector_expenditure
),
totals AS (
    SELECT source_name, year, price_basis, accounting_basis, unit_amount,
        COUNT(*) AS total_rows, MAX(amount) AS grand_total
    FROM amounts
    WHERE UPPER(TRIM(sector)) = 'GRAND TOTAL'
    GROUP BY source_name, year, price_basis, accounting_basis, unit_amount
),
compared AS (
    SELECT a.*,
        CASE WHEN t.total_rows = 1 AND t.grand_total > 0
            THEN ABS(a.amount / t.grand_total * 100 - a.reported_pct)
        END AS gap
    FROM amounts a
    LEFT JOIN totals t
      ON a.source_name <=> t.source_name
     AND a.year <=> t.year
     AND a.price_basis <=> t.price_basis
     AND a.accounting_basis <=> t.accounting_basis
     AND a.unit_amount <=> t.unit_amount
)
SELECT COUNT(*) AS total_rows,
    COUNT(gap) AS evaluated_rows,
    COUNT_IF(gap IS NULL) AS not_evaluated_rows,
    COUNT_IF(gap > 0.01) AS percentage_mismatches,
    ROUND(MAX(gap), 6) AS largest_gap_percentage_points
FROM compared;


In [0]:
%sql
WITH amounts AS (
    SELECT *,
        TRY_CAST(REPLACE(TRIM(expenditure_amount), ',', '') AS DECIMAL(20, 2)) AS amount,
        TRY_CAST(REPLACE(TRIM(percent_distribution), '%', '') AS DECIMAL(10, 4)) AS reported_pct
    FROM pondolens.dev_bronze.sector_expenditure
)
SELECT source_name, year, price_basis, accounting_basis, unit_amount, sector,
    SUM(CASE WHEN UPPER(TRIM(particulars)) = 'ALL' THEN amount END) AS sector_total,
    SUM(CASE WHEN UPPER(TRIM(particulars)) <> 'ALL' THEN amount END) AS subsector_sum,
    sector_total - subsector_sum AS difference_million_pesos,
    COUNT_IF(amount IS NULL) AS unparsed_amount_rows
FROM amounts
WHERE NULLIF(TRIM(particulars), '') IS NOT NULL
GROUP BY source_name, year, price_basis, accounting_basis, unit_amount, sector
HAVING ABS(difference_million_pesos) > 1 OR COUNT_IF(amount IS NULL) > 0
ORDER BY source_name, year, sector;


## 4. Q1 records
Use actual rows to describe actual spending. This filter does not remove program/proposed rows from Silver.

The result includes totals and subsectors. Select one level before adding amounts.



In [0]:
%sql
SELECT
    year, sector, particulars, expenditure_amount, unit_amount, percent_distribution, source_name
FROM pondolens.dev_bronze.sector_expenditure
WHERE LOWER(TRIM(price_basis)) = 'actual'
ORDER BY year, sector, particulars;


## 5. Findings and proposed Silver treatments

Counts below cover all **414 Bronze rows**, unless the actual-only count is stated.

| Check | Finding | Proposed Silver treatment |
|---|---|---|
| Coverage | Actual, program and proposed each have 138 rows. Actual covers 2021–2025; other stages extend through 2027. | Load only the 138 actual records into Silver. Retain all stages in Bronze. |
| Types | Source fields are strings; year/page parsing passes. Required labels are present; no rescued rows. | Cast year to INT, expenditure_amount to DECIMAL(20,2), and percent_distribution to DECIMAL(10,4). Source_page can remain metadata or become INT. |
| Units | Every expenditure amount is in million pesos; accounting is cash-based. Percent_distribution is a percentage, not money. | Keep the unit with the amount. If PHP is required, multiply by 1,000,000 and label the converted field clearly. |
| Particulars | 45 blanks in Bronze; 15 in actual rows. These are NET LENDING, DEBT-SERVICE-INTEREST PAYMENTS and GRAND TOTAL. | Consider if blanks shoulds be transformed to null or zero. |
| Dashes | 7 dash amounts and 3 dash percentages in Bronze. Actual rows contain 3 dash amounts and 1 dash percentage. | Keep Bronze unchanged. In Silver, convert to either zero or null depending on review. |
| Duplicate and total checks | No repeated record keys. Each source/year/stage/accounting/unit group has exactly one GRAND TOTAL. | No rows need removing based on these checks. |
| Percentages | 407 rows compared successfully; 7 could not be compared. No difference exceeds 0.01 percentage points; the largest is 0.004996. | Retain the reported percentages; no correction is indicated. |
| Sector totals | 2 complete groups have totals 2 million pesos higher than their detail sums. Another 7 groups contain dash amounts, including one with a −2-million-peso partial difference. | Preserve reported amounts since bronze accurately reflects source. Need further review if the discrepancy can be kept. |
| Metadata | source_name identifies the PDF; _source_file records the CSV path. | Retain these and other source/load metadata for actual records. |

### Record Anomalies

Difference = reported `All` amount − sum of numeric subsector amounts. All values below are in **million pesos**. Inspected the `All` row and every detail row.

| Source publication | Year / stage | Sector | All amount | Detail sum | Difference | Silver relevance |
|---|---|---|---|---|---|---|
| 02_sector_expenditure_2021.pdf | 2023 / proposed | GENERAL PUBLIC SERVICES | 807,158 | 807,156 | +2 | Source accurate and no problem in ingestion/transformation. No action needed since it's proposed and will be excluded from actual-only Silver. |
| 02_sector_expenditure_2022.pdf | 2022 / actual | ECONOMIC SERVICES | 1,687,362 | 1,687,360 | +2 | Source accurate and no problem in ingestion/transformation. Need further review.|
| 02_sector_expenditure_2023.pdf | 2023 / actual | SOCIAL SERVICES | 1,815,335 | 1,815,337 | −2 | Source accurate and no problem in ingestion/transformation. Need further review due to Partial sum: Land Distribution is a dash. |

## 6. Assumptions and open checks

| Question | Status |
|---|---|
| Does the Land Distribution dash mean zero, missing or not applicable? | Pending source review. This determines whether Silver uses zero or null. |
| Are the 2-million-peso differences explained by source rounding? | It is accurate based on source. Need further consultation. |
| Are actual, program and proposed budget stages to be used? | Actual-only Silver scope is needed for silver and disregard program/proposed. |

## 7. Semantic notes

- Amounts are cash-based expenditure in million pesos.
- Source publications are from 2021–2025. Program/proposed rows include expenditure years through 2027; keep them in Bronze and exclude them from Silver.
- A percentage shows how much a row contributes to the GRAND TOTAL from the same source, year and budget stage. For example, 25 means 25%, not 0.25%.
- Use sector totals or their details, not both. Do not add GRAND TOTAL to its components. DEFENSE All and Domestic Security represent the same amount in the earlier outputs.
- This table shows national sector spending. 

## 8. Conclusion

- Use only the **138 actual-spending records from 2021–2025** for Q1 and Silver. Retain program/proposed records in Bronze.
- Apply the numeric types in Silver: year as INT, amount as DECIMAL(20,2), and percentage as DECIMAL(10,4). Current queries only test parsing; they do not change stored types.
- Convert the **15 blank particulars in actual records** to null. Keep the records.
- Preserve the **7 dash amounts and 3 dash percentages in Bronze**. For Silver's actual rows (**3 amounts and 1 percentage**), choose zero or null after source review; use null while unresolved.
- Consult for the 2022 ECONOMIC SERVICES difference and the incomplete 2023 SOCIAL SERVICES sum. Keep reported amounts until a correction is supported.
- Retain source/load metadata so each Silver record can be traced to its source. When calculating totals, use either sector totals or their subsectors to avoid counting the same spending twice.
- Keep amounts labelled as million pesos, or convert to PHP when required. 



In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.sector_expenditure
WHERE year = 2023
  AND LOWER(TRIM(price_basis)) = 'actual'
  AND LOWER(TRIM(sector)) = 'social services';